# Day 3-02｜YOLO Tracking 與 ByteTrack ID
> Python 籃球運動資料分析課程  
> 本單元使用 Ultralytics `model.track(..., tracker="bytetrack.yaml")` 對實際籃球影片產生 track ID。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 執行 YOLO tracking mode。
- 讀取 ByteTrack 產生的 `track_id`。
- 輸出含 track ID 的短片段與 JSON。


## 課程流程
1. 選擇參考影片與 detector 權重。
2. 使用 `bytetrack.yaml` 執行 tracking。
3. 儲存 track records 與 annotated preview video。


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO

from src.cv_utils import save_json
from src.yolo_utils import (
    DetectionRecord,
    detector_model_path,
    draw_detection_records,
    first_reference_video,
    mp4_fourcc,
    records_to_dicts,
)

VIDEO_PATH = first_reference_video(COURSE_ROOT)
MODEL_PATH = detector_model_path(COURSE_ROOT)
MAX_FRAMES = 120

model = YOLO(str(MODEL_PATH))
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

out_video = COURSE_ROOT / "assets" / "results" / "d3_02_bytetrack_preview.mp4"
out_video.parent.mkdir(parents=True, exist_ok=True)
writer = cv2.VideoWriter(str(out_video), mp4_fourcc(), fps, (width, height))
records = []

for frame_index, result in enumerate(
    model.track(
        source=str(VIDEO_PATH),
        tracker="bytetrack.yaml",
        stream=True,
        persist=True,
        conf=0.25,
        imgsz=960,
        verbose=False,
    )
):
    if frame_index >= MAX_FRAMES:
        break
    frame_rgb = cv2.cvtColor(result.orig_img, cv2.COLOR_BGR2RGB)
    boxes = result.boxes
    frame_records = []
    if boxes is not None:
        names = result.names or {}
        for i in range(len(boxes)):
            box = boxes[i]
            track_id = None
            if getattr(boxes, "id", None) is not None:
                track_id = int(boxes.id[i].item())
            class_id = int(box.cls[0].item())
            det = DetectionRecord(
                frame_index=frame_index,
                class_id=class_id,
                class_name=str(names.get(class_id, class_id)),
                confidence=float(box.conf[0].item()),
                bbox_xyxy=[float(v) for v in box.xyxy[0].tolist()],
                track_id=track_id,
            )
            frame_records.append(det)
    vis = draw_detection_records(frame_rgb, frame_records, max_items=25)
    writer.write(cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
    records.extend(records_to_dicts(frame_records))

writer.release()
out_json = COURSE_ROOT / "assets" / "results" / "d3_02_tracks.json"
save_json(records, out_json)

print("video:", VIDEO_PATH)
print("preview video:", out_video)
print("tracks json:", out_json)
print("records:", len(records))
pd.DataFrame(records).head()


`track_id` 是後續 BEV 路徑與戰術資料表的主鍵。若 detector 漏偵、重複框過多或遮擋時間過長，ID 仍可能中斷或交換。